In [ ]:
import math
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from tqdm import tqdm

import torch
import torch.nn.functional as F

cm = ListedColormap(
    np.fromfile(
        '../data/colorbar/colorbar_ind.r@',
        dtype=np.float32,
    ).reshape(3, 256).T
)

sys.path.append('../')

from package.networks.unet import UNetModel


def show_batch_row(tensor, savepath, savename, cmap='jet'):
    if tensor.dim() == 4:
        tensor = tensor[:, 0, :, :]
    if tensor.dim() == 5:
        tensor = tensor[:, 0, 0, :, :]

    batch_size, _, _ = tensor.shape
    data = tensor.detach().cpu().numpy()
    fig, axes = plt.subplots(1, batch_size, figsize=(3 * batch_size, 3))
    if batch_size == 1:
        axes = [axes]

    for index in range(batch_size):
        axes[index].imshow(
            data[index],
            cmap=cmap,
            aspect=0.3,
            origin='upper',
            vmin=-0.7,
            vmax=0.6,
        )
        axes[index].axis('off')

    plt.savefig(f'{savepath}/{savename}', dpi=600)
    plt.close(fig)


def denormalize(x_norm, min_val=1000, max_val=5000):
    return (x_norm + 1) / 2 * (max_val - min_val) + min_val


def plot_formal(
    tensor,
    save_path,
    well_log_start_depth=None,
    well_log_locations=None,
    save_name='result',
):
    tensor = denormalize(tensor)
    data = tensor.detach().cpu().numpy().squeeze()

    dh = 12.5 * 2
    extent = [
        275 * dh / 1000,
        (275 + data.shape[1]) * dh / 1000,
        data.shape[0] * dh * 0.5 / 1000,
        0,
    ]

    vmin, vmax = 1000, 4800
    fig, ax = plt.subplots(figsize=(10 / 1.5, 5 / 1.5))
    im = ax.imshow(
        data,
        cmap=cm,
        vmin=vmin,
        vmax=vmax,
        aspect='auto',
        extent=extent,
    )

    if well_log_locations is not None:
        for index, well_x in enumerate(well_log_locations):
            x_km = (275 + well_x) * dh / 1000
            z_start_km = well_log_start_depth[index] * dh * 0.5 / 1000

            ax.plot(
                [x_km, x_km],
                [z_start_km, extent[2]],
                linestyle='--',
                linewidth=2.5,
                color='white',
            )

    cbar = plt.colorbar(im)
    cbar.set_label('Velocity (m/s)')

    ax.set_xlabel('Distance (km)', fontsize=12)
    ax.set_ylabel('Depth (km)', fontsize=12)
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_yticks([0, 1, 2, 3, 4])
    ax.set_yticklabels(['0.0', '1.0', '2.0', '3.0', '4.0'])

    plt.tight_layout()
    plt.savefig(
        f'{save_path}/{save_name}',
        dpi=600,
        bbox_inches='tight',
    )
    plt.close(fig)


def visualize_sample(data, save_path, filename, ncols=5):
    """
    Visualize N seismic samples automatically in a grid.

    Parameters
    ----------
    data : numpy.ndarray
        Shape can be [N, 1, 1, H, W], [N, 1, H, W], or [N, H, W].
    save_path : str
        Directory where the figure will be saved.
    filename : str
        Output filename.
    ncols : int
        Number of samples per row.
    """
    data = denormalize(data)
    data = np.squeeze(data)

    if data.ndim == 2:
        data = data[np.newaxis, ...]

    n_samples = data.shape[0]
    ncols = min(ncols, n_samples)
    nrows = math.ceil(n_samples / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(3.5 * 1.75 * ncols, 3.5 * nrows),
        squeeze=False,
    )
    axes = axes.flatten()

    for index in range(n_samples):
        axes[index].imshow(
            data[index],
            cmap='jet',
            aspect='auto',
            vmin=1000,
            vmax=4800,
        )
        axes[index].set_title(f'Sample {index + 1}')
        axes[index].axis('off')

    for index in range(n_samples, len(axes)):
        axes[index].axis('off')

    plt.tight_layout()
    plt.savefig(
        f'{save_path}/{filename}',
        dpi=300,
        bbox_inches='tight',
    )
    plt.close(fig)

## Model Initialization and Configuration

This cell loads the pretrained U-Net checkpoint used for prior injection with the Flow Matching model.

The model runs in inference mode, and all parameters are frozen to prevent weight updates during generation. This is a class-conditional Flow Matching model with two learned prior classes. Select the prior class in the sampling session before running the generation.

In [ ]:
batch_size = 1
use_cfg = False
device = 'cuda'
checkpoint_path = './checkpoints/unet_110.pth' 


def print_config(**kwargs):
    width = max(len(key) for key in kwargs) + 2
    print('\n' + '=' * 50)
    print(' Model Configuration '.center(50, '='))
    print('=' * 50)
    for key, value in kwargs.items():
        print(f'{key:<{width}}: {value}')
    print('=' * 50 + '\n')


print_config(
    use_cfg=use_cfg,
    device=device,
    checkpoint_path=checkpoint_path,
)

model = UNetModel(
    image_size=320,
    in_channels=1,
    out_channels=1,
    num_classes=2,
    model_channels=192,
    channel_mult=(1, 2, 4, 8),
    num_res_blocks=2,
    attention_resolutions=[80, 40],
    num_head_channels=64,
    dropout=0.05,
    use_scale_shift_norm=True,
    resblock_updown=True,
)

model.to(device)
model.eval()

checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model'])
for parameter in model.parameters():
    parameter.requires_grad = False

## Dataset Preparation

This cell loads the Viking FWI result and prepares it at the spatial resolution and field-of-view used by the model.

In [ ]:
fwi_velocity = np.load('../data/viking_data/ifwi_viking_ep400.npy')
fwi_velocity = 2 * (fwi_velocity - 1000) / (5000 - 1000) - 1
fwi_velocity = fwi_velocity[::2, ::2]
fwi_velocity = fwi_velocity[:320, 275:275 + 608]

print(f'fwi_velocity shape: {fwi_velocity.shape}')

## Well-Log Preparation

This cell loads the two well logs used for optional well-log guidance and converts their coordinates to the input-model reference frame.

In [ ]:
wells = torch.from_numpy(
    np.load('../data/well_logs/well_vp_interp.npy')[:, ::2]
).float().to(device)

wells = 2 * (wells - 1000) / (5000 - 1000) - 1

well_log_locations_ori = [
    int(807 / 2),
    int(1571 / 2),
]  # Positions of the well logs in the loaded model.

well_log_locations_input = [
    location - 275 for location in well_log_locations_ori
]

well_log_start_depth = 100  # Ignore the first 100 depth samples.

## Posterior Sampling with Observation-Guided Flow Matching

This cell implements the guided sampling procedure used for conditional velocity model generation.

The sampling process combines the learned seismic prior from the pretrained Flow Matching model with external observational constraints, primarily the velocity model obtained from Full Waveform Inversion (FWI). Optional well-log information can also be incorporated as an additional constraint.

A Gaussian smoothing operator is constructed and applied to the velocity model predicted by the Flow Matching model at each sampling step. The smoothed prediction is compared with the FWI result to enforce consistency with the large-scale structures captured by FWI.

This smoothing-based observation constraint allows the generated model to remain consistent with the large-scale FWI structure while retaining the finer-scale structural information provided by the learned seismic prior.

A total variation (TV) regularization term is additionally included to promote spatial smoothness and stabilize the guided sampling process.

The generation starts from an intermediate state obtained by combining the FWI observation with random Gaussian noise. The starting point is controlled by `start_time`, allowing the effect of observation guidance at different stages of the Flow Matching trajectory to be investigated.

At each ODE sampling step, the pretrained Flow Matching model predicts the transport velocity field. The model prediction is also used to estimate the corresponding terminal clean velocity model, which is then used to evaluate the observation-guidance losses.

Three loss terms can be used to provide guidance:

- **FWI consistency loss:** measures the discrepancy between the Gaussian-smoothed predicted velocity model and the FWI velocity model.

- **Well-log consistency loss:** optionally enforces agreement between the predicted velocity model and the available well-log values at selected spatial locations.

- **TV regularization loss:** promotes spatial smoothness of the predicted velocity model.

The weighted sum of these terms defines the guidance objective. Its gradient with respect to the current Flow Matching state is normalized and used to update the state before the subsequent ODE transport step.

The process can therefore be viewed as an alternating procedure:

```text
Flow Matching prediction
        ↓
Predicted terminal velocity model
        ↓
Observation-guidance losses
        ↓
Gradient-based state update
        ↓
Flow Matching ODE transport
        ↓
Next sampling state

In [ ]:
def gaussian_kernel2d(sigma, device):
    """Create a normalized 2D Gaussian kernel."""
    radius = int(3 * sigma)
    ksize = 2 * radius + 1

    ax = torch.arange(-radius, radius + 1, device=device)
    xx, yy = torch.meshgrid(ax, ax, indexing='ij')

    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()

    return kernel.view(1, 1, ksize, ksize)


def gaussian_smooth_2d(x, sigma=3):
    """Apply Gaussian smoothing with reflection padding."""
    kernel = gaussian_kernel2d(sigma, x.device)
    pad = kernel.shape[-1] // 2

    x_pad = F.pad(
        x,
        (pad, pad, pad, pad),
        mode='reflect',
    )

    return F.conv2d(x_pad, kernel)


def tv_loss(x):
    """Compute the total variation regularization loss."""
    dx = x[..., 1:] - x[..., :-1]
    dy = x[:, :, 1:, :] - x[:, :, :-1, :]

    return dx.abs().mean() + dy.abs().mean()


def bandpass_filter_torch(f_low, f_high, x, dt, pad=(125, 125), order=8):
    """Apply a differentiable Butterworth band-pass filter.

    Parameters
    ----------
    f_low : float
        High-pass cutoff frequency.
    f_high : float
        Low-pass cutoff frequency.
    x : torch.Tensor
        Input tensor with shape (..., nt).
    dt : float
        Temporal sampling interval.
    pad : tuple[int, int], optional
        Number of samples to pad on the left and right.
    order : int, optional
        Butterworth filter order.
    """
    original_shape = x.shape
    x = x.reshape(1, 1, -1)

    if pad is not None:
        x = F.pad(x, pad, mode='reflect')

    nt = x.shape[-1]
    device = x.device
    freqs = torch.fft.fftfreq(nt, d=dt).to(device).abs()
    eps = 1e-12

    # High-pass response.
    H_hp = 1.0 / torch.sqrt(
        1.0 + (f_low / (freqs + eps)) ** (2 * order)
    )

    # Low-pass response.
    H_lp = 1.0 / torch.sqrt(
        1.0 + (freqs / (f_high + eps)) ** (2 * order)
    )

    # Combine the responses into a band-pass filter.
    H = H_hp * H_lp
    X = torch.fft.fft(x, dim=-1)
    Xf = X * H
    x = torch.fft.ifft(Xf, dim=-1).real

    if pad is not None:
        left, right = pad
        x = x[..., left:-right]

    return x.reshape(original_shape)


# Sampling configuration.
ODE_step = 100
start_time = 30  # Try different values to study guidance at different stages.

dt = 1.0 / ODE_step
prior_type = 'Otway'  # Choose either 'Otway' or 'CGG'.

if prior_type == 'Otway':
    type = 0  # 0: Otway prior; 1: CGG prior.
else:
    type = 1
    assert prior_type == 'CGG', (
        "Invalid prior type. Must be 'Otway' or 'CGG'."
    )


# Guidance configuration and output path.
fwi_constraint = 1
well_constraint = 0  # Set to 0.5 to enable well-log guidance.
tv_constraint = 0.005
option = 'post_fwi_viking'
index_f = (
    '/FWI_guidance/'
    # Use '/FWI_well_joint_guidance/' to enable well-log guidance.
)
save_path = '../results/' + option + index_f + '/' + prior_type

os.makedirs(save_path, exist_ok=True)


# Prepare the conditioning and observation tensors.
y = torch.tensor([type]).long().to(device)
x_obs = (
    torch.from_numpy(fwi_velocity)
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device=device, dtype=torch.float32)
)

assert len(x_obs.shape) == 4, (
    'x_obs must be a 4D tensor with shape '
    '(batch_size, channels, height, width)'
)

seeds = range(2, 3)  # Extend the range to generate samples with more seeds.
posterior = []

for seed in seeds:
    print(f'Generating sample with seed: {seed}')

    # Random initialization.
    g = torch.Generator(device=device).manual_seed(seed)
    noise = torch.randn(
        x_obs.shape,
        generator=g,
        device=device,
    )

    x_t = (
        (start_time / ODE_step) * x_obs
        + (1 - (start_time / ODE_step)) * noise
    )

    mask = torch.ones_like(x_t)
    latent = []
    loss_latent = []

    # Reverse ODE sampling.
    batch_bar = tqdm(
        range(start_time, ODE_step - 6),
        desc='Generation steps',
        leave=False,
    )

    for index, j in enumerate(batch_bar):
        j = int(j)

        # Control the guidance learning rate during reverse sampling.
        ratio = (ODE_step - j) / ODE_step
        lr = 2.5e0 * ratio**1.25

        x_t = x_t.detach().requires_grad_(True)
        t = torch.tensor([j * dt], device=device)

        # Flow Matching prediction.
        v_pred = model(
            x=x_t,
            timesteps=t,
            y=y,
        )
        x11 = x_t + (1 - t) * v_pred

        # Save intermediate states for visualizing the generation process.
        if j % 10 == 0:
            latent.append(x11.detach().clone())

        x11 = x11.clamp(-1, 1)

        # Observation guidance.
        x_sys1 = gaussian_smooth_2d(x11[..., :110, :], sigma=0.1)
        x_sys2 = gaussian_smooth_2d(x11[..., 110:, :], sigma=3)
        x_sys = torch.cat([x_sys1, x_sys2], dim=2)

        loss_fwi = (
            F.mse_loss(
                x_sys,
                x_obs,
                reduction='none',
            ) * mask
        ).mean()

        # Use high-frequency well-log information after the first 100 depth
        # samples, where the interpolated values have limited resolution.
        loss_well = sum(
            F.mse_loss(
                bandpass_filter_torch(
                    0.025,
                    500,
                    x11[..., well_log_start_depth:, well_log_input_i],
                    1,
                ),
                bandpass_filter_torch(
                    0.025,
                    500,
                    wells[..., well_log_start_depth:, well_log_ori_i],
                    1,
                ),
                reduction='mean',
            )
            for well_log_input_i, well_log_ori_i in zip(
                well_log_locations_input,
                well_log_locations_ori,
            )
        )

        loss_tv = tv_loss(x11)
        loss = (
            fwi_constraint * loss_fwi
            + well_constraint * loss_well
            + tv_constraint * loss_tv
        )

        # Gradient guidance.
        grad = torch.autograd.grad(loss, x_t)[0]
        grad = grad / (grad.norm() + 1e-8)

        with torch.no_grad():
            x_t -= lr * grad

        # ODE transport step.
        x_t = x_t + v_pred * dt
        loss_latent.append(loss.item())

        batch_bar.set_postfix({
            'Loss_fwi': f'{loss_fwi.item():.6f}',
            'Loss_well': f'{loss_well.item():.6f}',
        })

    posterior.append(x11)


# Stack intermediate states and final samples along the batch dimension.
generation_tensor = torch.stack(
    latent,
    dim=0,
)
posterior_tensor = torch.stack(
    posterior,
    dim=0,
)

## Save Visualization Results

Run the final visualization cell after posterior sampling. It saves the guided result, original FWI observation, smoothed observation, generation trajectory, and posterior samples to `save_path`.

When well-log guidance is enabled, the guided result is also annotated with the well-log locations.

In [ ]:
# Save the generated models and diagnostic figures.
if 'well' in index_f:
    plot_formal(
        x11,
        save_path,
        well_log_start_depth=[well_log_start_depth, well_log_start_depth],
        well_log_locations=well_log_locations_input,
        save_name='prior_injected_result.png',
    )
else:
    plot_formal(
        x11,
        save_path,
        save_name='prior_injected_result.png',
    )

plot_formal(
    x_obs,
    save_path,
    save_name='original_fwi.png',
)
plot_formal(
    x_sys,
    save_path,
    save_name='smoothed_observation.png',
)

show_batch_row(
    generation_tensor,
    save_path,
    'generation_process.png',
)
visualize_sample(
    posterior_tensor.detach().cpu().numpy(),
    save_path,
    'posterior_samples.png',
)